# Task Hive — Colab 게이트웨이 노트북

이 노트북은 **내 구글 계정의 Colab(GPU)** 에서 대화 모델을 띄우고, Task Hive가 붙을 수 있는
**OpenAI 호환 게이트웨이**를 ngrok 주소로 열어 준다.

```
Task Hive 서버 ──(ngrok 주소 + 접속 키)──▶ 게이트웨이(:8000) ──▶ vLLM(:8001) ──▶ 모델
```

**쓰는 범위 (꼭 읽기)**

- 이 노트북은 **과제용 실험 환경**이다. LLM이 한국어 대화를 요약하고 할 일을 뽑을 수 있는지 확인하는 데 쓴다.
- Colab 무료는 자원이 보장되지 않고, 웹 서비스처럼 계속 켜 두는 용도를 제한한다. 세션이 경고 없이 끊길 수 있다.
  (`docs/checks/colab-usage.md`)
- 본인이 켜 두고 본인이 시연할 때만 쓴다. 다 쓰면 맨 아래 "끄기" 셀을 실행한다.

**저장소에 올리기 전에**: 메뉴의 `수정 → 출력 모두 지우기`를 한다. 출력에 ngrok 주소와 접속 키가 남는다.

**이 노트북의 상태**: 작성만 했고 Colab에서 실행해 확인하지 못했다. "확인 필요"라고 적힌 곳은
실행하면서 고쳐야 할 수 있다.

## 0. 설정

바꿀 값은 이 셀에만 있다. 아래 셀들은 이 변수를 읽는다.

**먼저 할 일 (한 번만)**: Colab 왼쪽 **열쇠 아이콘(보안 비밀)** 에 이름 `NGROK_AUTHTOKEN`(앞뒤 공백 없이)으로 토큰을 넣고 "노트북 액세스"를 켠다. 이 셀이 토큰을 먼저 읽어서, 값이 없거나 모양이 틀리면 vLLM 을 띄우기 전에 멈춘다.

| 변수 | 뜻 |
|---|---|
| `MODEL_NAME` | 띄울 모델. `llm-chatbot` 프로젝트에서 쓰던 모델이 기본값이다 |
| `MAX_MODEL_LEN` | 한 번에 다룰 수 있는 토큰 수(문맥 길이). 크게 잡을수록 GPU 메모리를 더 쓴다 |
| `GPU_MEMORY_UTILIZATION` | vLLM이 미리 잡아 둘 GPU 메모리 비율. 2단계에서 TTS·Whisper를 함께 올리려면 낮춰야 한다 |
| `NGROK_AUTHTOKEN` | 보안 비밀(왼쪽 열쇠 아이콘)에서 읽는다. ngrok 대시보드 **Your Authtoken**의 값만 넣는다(명령어 `ngrok config add-authtoken …` 통째로 넣지 않는다) |
| `NGROK_DOMAIN` | ngrok 무료 계정의 고정 도메인. 보안 비밀 `NGROK_DOMAIN`에 넣는다. 없으면 ngrok이 정한 주소를 쓴다 |

확인 필요: `MAX_MODEL_LEN` 과 `GPU_MEMORY_UTILIZATION` 의 알맞은 값은 받은 GPU에 따라 다르다.
PoC-03(`docs/poc/poc-03-colab-gpu-memory.md`)에서 재서 정한다.

In [ ]:
# ── 설정: 여기만 고친다 ─────────────────────────────────────────────
from google.colab import userdata
from google.colab.userdata import SecretNotFoundError, NotebookAccessError

MODEL_NAME = "cyankiwi/Qwen3.5-4B-AWQ-4bit"   # llm-chatbot 에서 쓰던 모델
MAX_MODEL_LEN = 8192                           # 문맥 길이(토큰). 저녁 대화 한 번이면 충분하다고 보고 잡은 값(확인 필요)
GPU_MEMORY_UTILIZATION = 0.80                  # vLLM 이 잡을 GPU 메모리 비율(확인 필요)
VLLM_PORT = 8001                               # vLLM 은 Colab 안에서만 접근한다(밖으로 열지 않는다)
GATEWAY_PORT = 8000                            # ngrok 으로 여는 것은 이 포트뿐이다


def read_secret(name, required=True):
    """Colab 보안 비밀을 읽는다. 앞뒤 공백·줄바꿈은 지운다."""
    try:
        return (userdata.get(name) or "").strip()
    except (SecretNotFoundError, NotebookAccessError):
        if required:
            raise RuntimeError(
                f"보안 비밀 '{name}'을 읽지 못했다. 이름(앞뒤 공백 없이)과 '노트북 액세스' 스위치를 확인한다"
            ) from None
        return ""


# 보안 비밀(왼쪽 열쇠 아이콘)에서 읽는다. 셀에 값을 직접 적지 않는다.
NGROK_AUTHTOKEN = read_secret("NGROK_AUTHTOKEN")             # dashboard.ngrok.com 의 Your Authtoken (명령어 말고 토큰만)
NGROK_DOMAIN = read_secret("NGROK_DOMAIN", required=False)   # 고정 도메인(예: 내이름.ngrok-free.app). 없으면 ngrok 이 정한다

if " " in NGROK_AUTHTOKEN or NGROK_AUTHTOKEN.startswith("ngrok"):
    raise RuntimeError("NGROK_AUTHTOKEN 값에 명령어나 공백이 들어 있다. 토큰만 넣는다")
# 토큰 값은 찍지 않는다. 읽었는지만 보여 준다.
print("설정 완료. ngrok 토큰 읽음:", bool(NGROK_AUTHTOKEN), "/ 고정 도메인:", NGROK_DOMAIN or "없음")

## 1. GPU 확인과 설치

- 먼저 메뉴 `런타임 → 런타임 유형 변경`에서 하드웨어 가속기를 **GPU**로 바꾼다.
- 설치는 몇 분 걸린다. 설치 뒤에 "런타임 다시 시작" 안내가 나오면 다시 시작하고 **0번 셀부터** 다시 실행한다.

확인 필요: `vllm` 은 버전에 따라 Colab의 CUDA·torch 와 맞지 않을 수 있다. 설치 오류가 나면
오류 내용에 나온 버전으로 맞춘다. `llm-chatbot` 문서에는 업스트림이 vLLM 0.33.0 이었다고 적혀 있다.

In [ ]:
# GPU 가 잡혔는지 확인한다. 표가 나오지 않으면 런타임 유형을 GPU 로 바꾸고 다시 실행한다.
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

# vllm: 모델을 OpenAI 호환 API 로 띄우는 서버
# fastapi, uvicorn, httpx: 접속 키를 확인하는 게이트웨이
# pyngrok: ngrok 터널을 파이썬에서 여는 도구
%pip install -q vllm fastapi "uvicorn[standard]" httpx pyngrok

## 2. vLLM 서버 실행 (백그라운드)

모델을 내려받고 GPU에 올린다. 처음에는 몇 분 걸린다. 로그는 `vllm.log` 파일에 쌓인다.

- `--host 127.0.0.1`: Colab 안에서만 접근하게 한다. 밖에서는 게이트웨이를 거쳐야만 닿는다.
- 생각 과정(thinking) 끄기는 여기서 하지 않는다. 요청마다 `chat_template_kwargs: {"enable_thinking": false}` 를
  보내야 하고, 게이트웨이가 이 값을 기본으로 넣어 준다(5번 셀).

확인 필요: 실행 옵션 이름은 vLLM 버전에 따라 다를 수 있다. 3번 셀에서 준비가 안 되면 `vllm.log` 를 본다.

In [ ]:
import subprocess, sys

# 이미 떠 있으면 다시 띄우지 않는다(셀을 두 번 실행해도 GPU 메모리를 두 번 잡지 않게).
if "vllm_proc" in globals() and vllm_proc.poll() is None:
    print("vLLM 이 이미 실행 중이다. 다시 띄우려면 맨 아래 '끄기' 셀을 먼저 실행한다.")
else:
    vllm_log = open("vllm.log", "w")
    vllm_proc = subprocess.Popen(
        [
            sys.executable, "-m", "vllm.entrypoints.openai.api_server",
            "--model", MODEL_NAME,
            "--host", "127.0.0.1",
            "--port", str(VLLM_PORT),
            "--max-model-len", str(MAX_MODEL_LEN),
            "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        ],
        stdout=vllm_log, stderr=subprocess.STDOUT,
    )
    print("vLLM 을 시작했다. 다음 셀에서 준비될 때까지 기다린다.")

## 3. 준비될 때까지 기다리기

`/v1/models` 가 답할 때까지 5초마다 확인한다. 최대 15분 기다린다.
중간에 vLLM 이 죽으면 로그 끝부분을 보여 주고 멈춘다.

In [ ]:
import time, httpx

deadline = time.time() + 15 * 60
ready = False
while time.time() < deadline:
    if vllm_proc.poll() is not None:        # 프로세스가 끝났다 = 실패
        print("vLLM 이 종료됐다. 로그 끝부분:")
        !tail -n 30 vllm.log
        break
    try:
        r = httpx.get(f"http://127.0.0.1:{VLLM_PORT}/v1/models", timeout=5)
        if r.status_code == 200:
            ready = True
            print("준비됨. 모델:", [m["id"] for m in r.json().get("data", [])])
            break
    except Exception:
        pass                                 # 아직 뜨는 중이다
    time.sleep(5)

if not ready and vllm_proc.poll() is None:
    print("15분 안에 준비되지 않았다. vllm.log 를 확인한다.")

## 4. 접속 키 만들기

ngrok 주소를 아는 사람이 누구나 내 Colab의 모델을 쓰지 못하게 **접속 키**를 둔다(결정 D-22).

- 키는 노트북을 실행할 때마다 새로 만든다. 파일에 저장하지 않는다.
- 이 키를 Task Hive 오른쪽 상태창의 "접속 키" 칸에 넣는다.
- 키는 비밀번호처럼 다룬다. 채팅이나 저장소에 붙여 넣지 않는다.

In [ ]:
import secrets

# 이미 만든 키가 있으면 그대로 쓴다(셀을 다시 실행해도 Task Hive 에 넣은 키가 바뀌지 않게).
if "GATEWAY_KEY" not in globals():
    GATEWAY_KEY = secrets.token_urlsafe(32)
print("접속 키를 만들었다. 6번 셀에서 주소와 함께 보여 준다.")

## 5. 게이트웨이 실행

게이트웨이는 작은 FastAPI 서버다. 하는 일은 세 가지다.

1. 요청의 `Authorization: Bearer <접속 키>` 를 확인한다. 틀리면 401.
2. 맞으면 요청을 vLLM 으로 넘기고 답을 그대로 돌려준다(스트리밍 포함).
3. 요청에 `chat_template_kwargs` 가 없으면 `{"enable_thinking": false}` 를 넣는다.
   이 모델은 생각 과정을 먼저 쓰는 모델이라, 끄지 않으면 답이 비어 나올 수 있다(`llm-chatbot` 의 알려진 이슈).

| 주소 | 설명 |
|---|---|
| `GET /health` | Task Hive 가 "이 주소가 Task Hive 게이트웨이가 맞는지" 확인할 때 부른다 |
| `GET /v1/models` | 모델 목록 |
| `POST /v1/chat/completions` | 대화. OpenAI 호환 |
| `POST /tts`, `POST /stt` | 2단계에 만든다. 지금은 501 을 돌려준다 |

아래 첫 셀은 `gateway.py` 파일을 만들고, 둘째 셀이 그 파일을 백그라운드로 실행한다.

In [ ]:
%%writefile gateway.py
"""Task Hive 게이트웨이 (Colab 안에서 실행).

하는 일: 접속 키를 확인한 뒤 요청을 vLLM 으로 넘긴다.
왜: vLLM 을 그대로 ngrok 으로 열면 주소를 아는 누구나 쓸 수 있다.
아직 안 된 것: /tts, /stt (2단계).
"""
import hmac
import json
import os

import httpx
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, StreamingResponse

# 설정은 환경변수로 받는다(노트북 셀이 넣어 준다). 코드에 키를 적지 않는다.
GATEWAY_KEY = os.environ["TH_GATEWAY_KEY"]
VLLM_URL = os.environ.get("TH_VLLM_URL", "http://127.0.0.1:8001")
MAX_BODY_BYTES = 512 * 1024      # 요청 본문 한도. 아주 큰 요청으로 메모리를 채우는 것을 막는다.

# docs_url=None: 자동 문서 화면(/docs)을 끈다. 밖에서 API 구조를 훑지 못하게.
app = FastAPI(title="task-hive-gateway", docs_url=None, redoc_url=None, openapi_url=None)


def authorized(request: Request) -> bool:
    """Authorization: Bearer <키> 가 맞는지 본다.

    hmac.compare_digest 를 쓰는 이유: 일반 == 비교는 앞 글자가 맞을수록 조금 더 오래 걸려서
    걸린 시간으로 키를 한 글자씩 맞혀 볼 수 있다. 이 함수는 걸리는 시간이 일정하다.
    """
    header = request.headers.get("authorization", "")
    if not header.lower().startswith("bearer "):
        return False
    return hmac.compare_digest(header[7:].strip().encode(), GATEWAY_KEY.encode())


def deny() -> JSONResponse:
    # 왜 틀렸는지 자세히 알려 주지 않는다.
    return JSONResponse({"error": "unauthorized"}, status_code=401)


@app.get("/health")
async def health(request: Request):
    """Task Hive 가 연결 확인에 쓰는 주소. 키가 맞아야 답한다."""
    if not authorized(request):
        return deny()
    models = []
    try:
        async with httpx.AsyncClient(timeout=10) as client:
            r = await client.get(f"{VLLM_URL}/v1/models")
            models = [m["id"] for m in r.json().get("data", [])]
    except Exception:
        pass   # 모델 서버가 아직 안 떴으면 빈 목록으로 답한다
    return {"service": "task-hive-gateway", "models": models, "tts": False, "stt": False}


@app.get("/v1/models")
async def models(request: Request):
    if not authorized(request):
        return deny()
    async with httpx.AsyncClient(timeout=10) as client:
        r = await client.get(f"{VLLM_URL}/v1/models")
    return JSONResponse(r.json(), status_code=r.status_code)


@app.post("/v1/chat/completions")
async def chat(request: Request):
    """대화 요청을 vLLM 으로 넘긴다. stream=true 면 조각을 받는 대로 흘려보낸다."""
    if not authorized(request):
        return deny()
    raw = await request.body()
    if len(raw) > MAX_BODY_BYTES:
        return JSONResponse({"error": "request too large"}, status_code=413)
    try:
        body = json.loads(raw)
    except json.JSONDecodeError:
        return JSONResponse({"error": "invalid json"}, status_code=400)
    if not isinstance(body, dict):
        return JSONResponse({"error": "invalid json"}, status_code=400)

    # 생각 과정을 기본으로 끈다. 요청이 직접 정했으면 그 값을 따른다.
    body.setdefault("chat_template_kwargs", {"enable_thinking": False})

    url = f"{VLLM_URL}/v1/chat/completions"
    if body.get("stream"):
        async def relay():
            # 스트리밍 동안 연결을 열어 두어야 해서 client 를 이 함수 안에서 만든다.
            async with httpx.AsyncClient(timeout=httpx.Timeout(300, connect=10)) as client:
                async with client.stream("POST", url, json=body) as upstream:
                    async for chunk in upstream.aiter_bytes():
                        yield chunk
        return StreamingResponse(relay(), media_type="text/event-stream")

    async with httpx.AsyncClient(timeout=httpx.Timeout(300, connect=10)) as client:
        r = await client.post(url, json=body)
    return JSONResponse(r.json(), status_code=r.status_code)


@app.post("/tts")
async def tts(request: Request):
    """말하기(TTS). 2단계에 만든다. 지금은 Task Hive 가 브라우저 내장 음성을 쓴다."""
    if not authorized(request):
        return deny()
    return JSONResponse({"error": "not implemented", "phase": 2}, status_code=501)


@app.post("/stt")
async def stt(request: Request):
    """받아쓰기(STT, Whisper). 2단계에 만든다."""
    if not authorized(request):
        return deny()
    return JSONResponse({"error": "not implemented", "phase": 2}, status_code=501)

In [ ]:
import os, subprocess, sys, time, httpx

if "gateway_proc" in globals() and gateway_proc.poll() is None:
    gateway_proc.terminate()          # gateway.py 를 고쳤을 수 있으니 다시 띄운다
    gateway_proc.wait(timeout=10)

# 키와 vLLM 주소는 환경변수로 넘긴다. 명령줄 인자로 넘기면 프로세스 목록에 보인다.
env = dict(os.environ, TH_GATEWAY_KEY=GATEWAY_KEY, TH_VLLM_URL=f"http://127.0.0.1:{VLLM_PORT}")
gateway_log = open("gateway.log", "w")
gateway_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "gateway:app", "--host", "127.0.0.1", "--port", str(GATEWAY_PORT)],
    env=env, stdout=gateway_log, stderr=subprocess.STDOUT,
)
time.sleep(3)

# 스스로 확인: 키 없이 부르면 401, 키를 넣으면 200 이어야 한다.
base = f"http://127.0.0.1:{GATEWAY_PORT}"
print("키 없이:", httpx.get(f"{base}/health").status_code, "(401 이어야 한다)")
r = httpx.get(f"{base}/health", headers={"Authorization": f"Bearer {GATEWAY_KEY}"})
print("키 넣고:", r.status_code, r.json() if r.status_code == 200 else "")

## 6. ngrok 으로 열기

토큰은 0번 설정 셀이 보안 비밀 `NGROK_AUTHTOKEN`에서 이미 읽어 두었다(준비 방법도 0번 셀 설명에 있다).
토큰을 셀에 직접 붙여 넣지 않는다. 노트북 파일에 남아 저장소에 올라갈 수 있다.

이 셀을 실행하면 **주소와 접속 키**가 나온다. 두 값을 Task Hive 오른쪽 상태창의 "연결" 구역에 넣고
"연결 확인"을 누른다.

In [ ]:
from pyngrok import ngrok

# 토큰은 0번 셀이 보안 비밀에서 읽어 둔 값을 쓴다.
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# 셀을 다시 실행했을 때 터널이 여러 개 생기지 않게 먼저 닫는다.
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

options = {"domain": NGROK_DOMAIN} if NGROK_DOMAIN else {}
tunnel = ngrok.connect(GATEWAY_PORT, "http", **options)
PUBLIC_URL = tunnel.public_url.replace("http://", "https://")   # Task Hive 는 https 주소만 받는다

print("=" * 60)
print("Task Hive 상태창 → 연결 구역에 아래 두 값을 넣는다")
print("  ngrok 주소 :", PUBLIC_URL)
print("  접속 키    :", GATEWAY_KEY)
print("=" * 60)
print("주의: 이 출력을 저장소에 올리지 않는다. 올리기 전에 '출력 모두 지우기'를 한다.")

## 7. 밖에서 닿는지 확인 + GPU 메모리 (PoC-03)

- 첫 부분: ngrok 주소로 짧은 대화를 한 번 보내 본다. 한국어 답이 나오면 된다.
  `ngrok-skip-browser-warning` 헤더는 ngrok 무료 주소의 경고 페이지를 건너뛰려고 넣는다.
- 둘째 부분: 지금 GPU 메모리를 얼마나 쓰는지 본다. 이 숫자를 `docs/poc/poc-03-colab-gpu-memory.md` 에 적는다.

In [ ]:
import httpx

headers = {"Authorization": f"Bearer {GATEWAY_KEY}", "ngrok-skip-browser-warning": "1"}
payload = {
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "안녕? 한 문장으로 인사해 줘."}],
    "max_tokens": 60,
    "temperature": 0.2,
}
r = httpx.post(f"{PUBLIC_URL}/v1/chat/completions", json=payload, headers=headers, timeout=120)
print("상태:", r.status_code)
if r.status_code == 200:
    print("답:", r.json()["choices"][0]["message"]["content"])

print()
print("GPU 메모리 (PoC-03 기록용):")
!nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free --format=csv

## 8. PoC-01: 가상 대화 20건으로 할 일 추출 재기

이 셀은 **노트북 안에서** 측정 스크립트를 돌린다. ngrok 을 거치지 않고 게이트웨이(`127.0.0.1:8000`)를 바로 부른다.

- 저장소를 내려받아 `eval/run_eval.py` 를 실행한다.
- 결과는 `task-hive/eval/results/` 에 `.json` 과 `.md` 로 생긴다. 왼쪽 파일 창에서 내려받아
  내 컴퓨터의 저장소 `eval/results/` 에 넣는다.
- 저장소를 내려받는 대신 왼쪽 파일 창에 `eval/` 폴더를 직접 올려도 된다. 그때는 경로만 맞춘다.

정답은 모델을 돌리기 전에 이미 `eval/samples/dialogs.jsonl` 에 적혀 있다. 결과를 보고 정답을 고치지 않는다.

In [ ]:
import os

# 저장소 내려받기(공개 저장소). 이미 있으면 최신으로 맞춘다.
if not os.path.isdir("task-hive"):
    !git clone -q https://github.com/eple-lab/task-hive.git
else:
    !git -C task-hive pull -q

# 키는 환경변수로 넘긴다. 명령줄에 적으면 출력에 남는다.
os.environ["TASK_HIVE_EVAL_API_KEY"] = GATEWAY_KEY
!cd task-hive && python eval/run_eval.py \
    --base-url http://127.0.0.1:{GATEWAY_PORT} \
    --model "{MODEL_NAME}" \
    --label qwen4b-colab \
    --completions eval/samples/completions.jsonl

## 9. 끄기

다 쓰면 실행한다. ngrok 터널을 닫고 게이트웨이와 vLLM 을 끈다.
그다음 메뉴 `런타임 → 런타임 연결 해제 및 삭제`로 GPU 를 돌려준다.

In [ ]:
from pyngrok import ngrok

for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)
ngrok.kill()

for name in ("gateway_proc", "vllm_proc"):
    proc = globals().get(name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=20)
        except Exception:
            proc.kill()
print("터널, 게이트웨이, vLLM 을 껐다.")